## Priprava podatkov
Poglejmo si kako napravimo tokene iz besed. 

- Token (žeton) je osnovna enota besedila, na katero računalnik razdeli besedilo pri obdelavi naravnega jezika. V našem primeru bo token predstavljal kar en znak v besedi. Pomembno: token $\neq$ beseda.

Bolj realistični primeri bi bili:
- računalništvo	$\rightarrow$ računal + ništvo
- programiranje $\rightarrow$ program + iranje
- najlepši $\rightarrow$ naj + lep + ši

In [1]:
import pickle
import numpy as np
import glob
import random

## Podatki
Na voljo imamo več besedil Ivana Cankarja. Na voljo so v `naloga_2/vaje/data/cankar`. Razdeljena so na drame in na prozo.

In [2]:
drame_txt = glob.glob("data/cankar/drame/*.txt")
proza_txt = glob.glob("data/cankar/proza/*.txt")

cankar_txt = drame_txt + proza_txt
random.shuffle(cankar_txt)

data = ""
for c_txt in cankar_txt:
    with open(c_txt, 'r') as f:
        data += f.read() + "\n\n"

In [3]:
print(data[:1000])

Za vozom

Francka ni zaspala dolgo v noč. Vse je bilo že tiho, nič se ni zgenilo v temi in skoro jo je bilo strah. Samo poredkoma je segel rezek glas iz noči — zaukal je fant na vasi, zapel je pesem, šel je zmerom dalje po klancu navzdol in pesem je umolknila, potopila se je v noč. Zalajal je pes pri štacunarju — začul je bil bogvekaj, šuštenje kostanja v bližini, praskanje miši v prodajalnici, fantovo pesem iz daljave, in vzdignil je glavo ter zavil z zategnjenim, cvilečim glasom, zacvilil še enkrat že v poluspanju in legel na plahto in zadremal ...

Odeta je bila Francka samo z rjuho, pa ji je bilo vroče in pot ji je tekel od čela dol po licih in je močil blazino. V sobi je bilo tema in vzduh je bil težak in vroč; zdelo se ji je, da bi se izhladilo, če bi odgrnila zeleno zagrinjalo pred oknom in bi zasijala mehka svetloba ponočnega neba. Iz teme je prihajalo enakomerno, trudno sopenje matere in sestre, ki sta spali na postelji in tiktakanje velike stenske ure se je časih hipoma oglas

In [4]:
print(f"Dolžina besedila v znakih: {len(data):,}")

Dolžina besedila v znakih: 2,483,110


In [5]:
# vsi različni znaki, ki se pojavijo v besedilu
chars = sorted(list(set(data)))
vocab_size = len(chars) # codebook

In [6]:
print(f"Vsi različni znaki: {''.join(chars)}")
print(f"Velikost besedišča: {vocab_size}")

Vsi različni znaki: 
 !"'(),-.0123456789:;?ABCDEFGHIJKLMNOPRSTUVWZabcdefghijklmnopqrstuvwxyzČčŠšŽž—
Velikost besedišča: 79


- Tokenizer (tokenizator): algoritem, ki razdeli besedilo na žetone in jih pretvori v identifikacijske številke (ID-je).
- Vocabulary (besedišče): zbirka žetonov, ki jih tokenizator prepoznava, pri čemer ima vsak žeton pripadajočo številsko identifikacijsko oznako (ID).

Enostavni primer tokenizatorja je preslikava znaka v celo število

## Naš tokenizer

<img src="images/tokenization.png" width="800"/>

In [7]:
from char_tokenizer import SimpleTokenizer

In [8]:
# preslikava iz celega števila v znak in obratno
tokenizer = SimpleTokenizer(chars)

In [9]:
tokenizer("Danes je lep dan za Praktikum Strojnega učenja!")

[26,
 46,
 59,
 50,
 64,
 1,
 55,
 50,
 1,
 57,
 50,
 61,
 1,
 49,
 46,
 59,
 1,
 71,
 46,
 1,
 38,
 63,
 46,
 56,
 65,
 54,
 56,
 66,
 58,
 1,
 40,
 65,
 63,
 60,
 55,
 59,
 50,
 52,
 46,
 1,
 66,
 73,
 50,
 59,
 55,
 46,
 2]

In [10]:
# naredimo učno in validacijsko množico
split_perc = 0.9
n = len(data)
train_data = data[:int(n * split_perc)]
val_data = data[int(n * split_perc):]

In [11]:
# enkodiramo obe množici v cela števila
train_ids = tokenizer.encode(train_data)
val_ids = tokenizer.encode(val_data)

print(f"Učna množica ima {len(train_ids):,} tokenov.")
print(f"Validacijska množica ima {len(val_ids):,} tokenov.")

Učna množica ima 2,234,799 tokenov.
Validacijska množica ima 248,311 tokenov.


In [12]:
# shranimo naše tokenizirano besedilo v binarne .npy datoteke
np.save("data/simple_tok_train.npy", np.array(train_ids, dtype=np.uint32))
np.save("data/simple_tok_val.npy", np.array(val_ids, dtype=np.uint32))

In [13]:
# shranimo tokenizer
with open("data/simple_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

Poglejmo si še kako to naredimo z obstoječo knjižnico. Uporabili bomo: https://github.com/openai/tiktoken.

## Huggingface tokenizer

In [14]:
from transformers import AutoTokenizer

/Users/jan/repositories/PSUF/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# https://huggingface.co/cjvt/GaMS-1B
tokenizer = AutoTokenizer.from_pretrained("cjvt/GaMS-1B") # obstajajo tudi drugi
tokenizer.save_pretrained("data/tokenizer_gams1b")

('data/tokenizer_gams1b/tokenizer_config.json',
 'data/tokenizer_gams1b/tokenizer.json')

In [16]:
print("Velikost besedišča:", tokenizer.vocab_size)

Velikost besedišča: 80000


In [17]:
token_to_int = tokenizer.get_vocab() # token -> ID

int_to_token = {}
for token_bytes, token_id in token_to_int.items():
    int_to_token[token_id] = token_bytes

In [18]:
all_ids = sorted(int_to_token.keys())

random_start = np.random.randint(0, len(all_ids) - 20)

# 20 zaporednih ID-jev in pripadajočih tokenov
for i in all_ids[random_start:random_start + 20]:
    print(i, repr(int_to_token[i]))

52844 '▁pritegnili'
52845 '▁Takih'
52846 'mpin'
52847 '▁domeno'
52848 '▁omiljeni'
52849 '▁Elektronski'
52850 '▁matematično'
52851 '/15'
52852 '▁prevar'
52853 '▁replace'
52854 '▁Zbiranje'
52855 '▁lastniških'
52856 '▁razlikovanja'
52857 '▁tamkajšnjega'
52858 '▁crac'
52859 '▁psihoterapev'
52860 '▁oliver'
52861 '▁Valencia'
52862 '▁turbulent'
52863 '▁polufinala'


In [19]:
tokenizer(["Danes je lep dan", "za", "Praktikum Strojnega učenja!"], padding=False)

{'input_ids': [[3496, 34, 4977, 756], [43], [28825, 46425, 5722, 11167, 7812, 68047]], 'attention_mask': [[1, 1, 1, 1], [1], [1, 1, 1, 1, 1, 1]]}

In [20]:
ids = tokenizer(data, add_special_tokens=False)["input_ids"]
train_ids = ids[:int(n * split_perc)]
val_ids = ids[int(n * split_perc):]

In [21]:
np.save("data/gams1b_tok_train.npy", np.array(train_ids, dtype=np.uint32))
np.save("data/gams1b_tok_val.npy", np.array(val_ids, dtype=np.uint32))

## Literatura
- https://www.dlib.si/details/URN:NBN:SI:DOC-4UJD2JYX
- https://lit.ijs.si/leposl.html
- https://github.com/karpathy/nanoGPT